# Train/Test Predictive Models With GraphEstimator + Graph Vectorizers

Thin notebook for fitting classical graph predictive models on persisted PCNs. It uses `abstractgraph_ml.estimators.GraphEstimator` with a selectable graph vectorizer such as `SimplexPairGraphVectorizer` or `NSPPK`.


## 1. Configure

In [13]:
try:
    from _bootstrap import configure_paths
except ImportError:
    from notebooks._bootstrap import configure_paths

PROJECT_ROOT = configure_paths(include=("abstractgraph_graphicalizer", "simplex_pair_vectorizer", "abstractgraph_ml"))

from enzyme_classification.modeling.datasets import load_graph_dataset_from_status
from enzyme_classification.modeling.results import persist_graph_estimator_results
from enzyme_classification.modeling.splits import make_grouped_train_test_split
from abstractgraph_graphicalizer.protein import ProteinLabelGraphicalizer
from abstractgraph_ml.estimators import GraphEstimator
from simplex_pair_vectorizer import SimplexPairGraphVectorizer

try:
    from nsppk import NSPPK
except ImportError:
    NSPPK = None

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
)


### Node Label Alphabet Options

`NODE_LABEL_ALPHABET` controls how amino-acid node labels are grouped before graph vectorization. The graph topology is unchanged; only the node label seen by the vectorizer changes.

- `aa20`: keeps the original 20 amino-acid residue labels. This is the most detailed setting and is the baseline for comparison.
- `hp2`: reduces residues to hydrophobic or polar. This tests whether very coarse chemistry is enough.
- `charge4`: separates positive, negative, neutral polar, and nonpolar residues. This emphasizes electrostatic and polarity differences.
- `physchem5`: groups residues into hydrophobic, polar uncharged, positive, negative, and special residues. This is a compact general-purpose physicochemical alphabet.
- `chem7`: uses more specific side-chain chemistry: acidic, basic, hydroxyl, amide, sulfur, aromatic, and aliphatic/special. This keeps enzyme-relevant chemistry more visible while still reducing sparsity.
- `dayhoff6`: groups residues by a classic substitution-style reduced alphabet. This emphasizes evolutionary exchangeability rather than hand-picked chemistry.


In [14]:

TASK = "A"  # "A" for enzyme vs non-enzyme, "B" for first-level EC class
RANDOM_STATE = 4
TEST_SIZE = 0.20
MIN_SAMPLES_PER_CLASS_FOR_EVALUATION = 5
FILTER_UNDER_SUPPORTED_CLASSES = False
SPLIT_GROUP_COLUMN = "uniprot_accessions"  # Keep all structures for the same UniProt protein sequence in one split.
SPLIT_GROUP_FALLBACK_COLUMNS = ["protein_name", "entity_id", "pdb_id"]
TRANSFORMER_KIND = "simplex_pair"  # "simplex_pair" or "nsppk"
NODE_LABEL_ALPHABET = "aa20"  # "aa20", "hp2", "charge4", "physchem5", "chem7", or "dayhoff6"
EDGE_DISTANCE_THRESHOLDS = None  # None => "contact"; e.g. [5.0] => "contact_close"/"contact_far".
BASE_ESTIMATOR_KIND = "linear_sgd"  # "linear_sgd" or "random_forest"


TASK_A_STATUS = PROJECT_ROOT / "data/processed/task_a/graph_build_status.tsv"
TASK_B_STATUS = PROJECT_ROOT / "data/processed/task_b/graph_build_status.tsv"

STATUS_PATH = TASK_A_STATUS if TASK.upper() == "A" else TASK_B_STATUS
RESULTS_ROOT = PROJECT_ROOT / "results" / f"task_{TASK.lower()}"
STATUS_PATH

PosixPath('/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/EnzymeClassification/data/processed/task_a/graph_build_status.tsv')

## 2. Load Persisted Graph Dataset

In [15]:
dataset = load_graph_dataset_from_status(STATUS_PATH, task=TASK)
graphs = dataset.graphs
y = dataset.labels
metadata = dataset.metadata

print(f"task={TASK}")
print(f"status_path={STATUS_PATH}")
print(f"graphs={len(graphs)} labels={len(y)}")
metadata[["sample_id", "pdb_id", "protein_name", "first_level_ec_numbers", "target", "graph_path"]].head()

task=A
status_path=/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/EnzymeClassification/data/processed/task_a/graph_build_status.tsv
graphs=70393 labels=70393


,sample_id,pdb_id,protein_name,first_level_ec_numbers,target,graph_path
0,10dc.A,10dc,GTPase HRas,3,enzyme,/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b6...
1,10dj.A,10dj,Tyrosine-protein kinase Fyn,2,enzyme,/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b6...
2,10dj.B,10dj,Tyrosine-protein kinase Fyn,2,enzyme,/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b6...
3,10en.A,10en,"Isoform 4 of Glutamate receptor ionotropic, NM...",,non_enzyme,/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b6...
4,10en.C,10en,"Isoform 4 of Glutamate receptor ionotropic, NM...",,non_enzyme,/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b6...


In [16]:
pd.Series(y, name="target").value_counts().sort_index()

target
enzyme        32206
non_enzyme    38187
Name: count, dtype: int64

## 3. Train/Test Split

In [17]:
split_result = make_grouped_train_test_split(
    graphs,
    y,
    metadata,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    min_samples_per_class_for_evaluation=MIN_SAMPLES_PER_CLASS_FOR_EVALUATION,
    filter_under_supported_classes=FILTER_UNDER_SUPPORTED_CLASSES,
    split_group_column=SPLIT_GROUP_COLUMN,
    split_group_fallback_columns=SPLIT_GROUP_FALLBACK_COLUMNS,
)

graphs_for_split = split_result.graphs_for_split
y_for_split = split_result.y_for_split
metadata_for_split = split_result.metadata_for_split
all_labels = split_result.all_labels
class_counts = split_result.class_counts
under_supported_classes = split_result.under_supported_classes
split_counts = split_result.split_counts
active_under_supported_classes = split_result.active_under_supported_classes
group_counts = split_result.group_counts
train_idx = split_result.train_idx
test_idx = split_result.test_idx
X_train = split_result.X_train
X_test = split_result.X_test
y_train = split_result.y_train
y_test = split_result.y_test
train_groups = split_result.train_groups
test_groups = split_result.test_groups
group_overlap = split_result.group_overlap
missing_from_train = split_result.missing_from_train
missing_from_test = split_result.missing_from_test
VALID_HELD_OUT_EVALUATION = split_result.valid_held_out_evaluation
split_strategy = split_result.split_strategy
stratified = split_result.stratified

print("class support")
print(class_counts)

if not under_supported_classes.empty:
    print(
        "\nSmoke-test warning: these classes have fewer than "
        f"{MIN_SAMPLES_PER_CLASS_FOR_EVALUATION} samples and cannot support a reliable held-out evaluation."
    )
    print(under_supported_classes)
if FILTER_UNDER_SUPPORTED_CLASSES and not under_supported_classes.empty:
    print(f"\nFiltered to classes with >= {MIN_SAMPLES_PER_CLASS_FOR_EVALUATION} samples: {all_labels}")

if split_result.stratified_group_error:
    print(
        "\nSplit warning: StratifiedGroupKFold failed "
        f"({split_result.stratified_group_error}); falling back to GroupShuffleSplit."
    )

print(
    f"\ntrain={len(X_train)} test={len(X_test)} "
    f"strategy={split_strategy} stratified={stratified} group_column={SPLIT_GROUP_COLUMN}"
)
print(f"groups: train={len(train_groups)} test={len(test_groups)} overlap={len(group_overlap)} total={group_counts.size}")
print("largest groups")
print(group_counts.head(10))
print("train labels")
print(pd.Series(y_train).value_counts().sort_index())
print("test labels")
print(pd.Series(y_test).value_counts().sort_index())

if missing_from_train or missing_from_test:
    print("\nEvaluation warning: this split is not a complete held-out multiclass evaluation.")
    if missing_from_train:
        print(f"classes missing from train: {missing_from_train}")
    if missing_from_test:
        print(f"classes missing from test: {missing_from_test}")

if not VALID_HELD_OUT_EVALUATION:
    print("Treat the metrics below as smoke-test diagnostics, not final model performance.")


class support
enzyme        32206
non_enzyme    38187
Name: count, dtype: int64

train=56314 test=14079 strategy=stratified_group_kfold_5 stratified=True group_column=uniprot_accessions
groups: train=5809 test=1457 overlap=0 total=7266
largest groups
uniprot_accessions:p01308    1032
uniprot_accessions:p61769     891
uniprot_accessions:p00734     801
uniprot_accessions:p00918     581
uniprot_accessions:p02766     556
uniprot_accessions:p56817     518
uniprot_accessions:p24941     464
uniprot_accessions:p68871     452
uniprot_accessions:p69905     452
uniprot_accessions:p04439     411
Name: count, dtype: int64
train labels
enzyme        25765
non_enzyme    30549
Name: count, dtype: int64
test labels
enzyme        6441
non_enzyme    7638
Name: count, dtype: int64


## 4. Define GraphEstimator Transformer


In [18]:
from sklearn.linear_model import SGDClassifier
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import Normalizer


if TRANSFORMER_KIND == "nsppk":
    if NSPPK is None:
        raise ImportError("TRANSFORMER_KIND='nsppk' requires the nsppk package on PYTHONPATH.")
    graph_vectorizer = NSPPK(
        radius=1,
        distance=4,
        connector=1,
        nbits=14,
        dense=False,
        parallel=True,
    )
elif TRANSFORMER_KIND == "simplex_pair":
    graph_vectorizer = SimplexPairGraphVectorizer(
        max_simplex_dimension=3,
        min_distance=0,
        max_distance=4,
        distance_mode="representative",
        distance_jitter="auto",
        pairing_mode="rooted",
        nbits=14,
        dense=False,
        include_self=True,
        cache_graphs=False,
        n_jobs=-1,
    )
else:
    raise ValueError(f"Unknown TRANSFORMER_KIND={TRANSFORMER_KIND!r}")

transformer = Pipeline(
    [
        (
            "protein_label_graphicalizer",
            ProteinLabelGraphicalizer(
                alphabet=NODE_LABEL_ALPHABET,
                node_label_key="label",
                edge_label_key="label",
                edge_distance_key="distance",
                edge_distance_thresholds=EDGE_DISTANCE_THRESHOLDS,
            ),
        ),
        ("graph_vectorizer", graph_vectorizer),
    ]
)

if BASE_ESTIMATOR_KIND == "linear_sgd":
    base_estimator = make_pipeline(
        Normalizer(norm="l2"),
        SGDClassifier(
            loss="hinge",
            alpha=1e-4,
            max_iter=5000,
            tol=1e-4,
            n_iter_no_change=20,
            random_state=RANDOM_STATE,
            class_weight="balanced",
            average=True,
            n_jobs=-1,
        ),
    )
elif BASE_ESTIMATOR_KIND == "random_forest":
    base_estimator = RandomForestClassifier(
        n_estimators=1000,
        max_features="sqrt",
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        class_weight="balanced_subsample",
        n_jobs=-1,
    )
else:
    raise ValueError(f"Unknown BASE_ESTIMATOR_KIND={BASE_ESTIMATOR_KIND!r}")
model = GraphEstimator(
    transformer=transformer,
    estimator=base_estimator,
    manifold=None,
    n_selected_features=None,
)
model


,transformer,"Pipeline(step..., nbits=14))])"
,estimator,Pipeline(step...tol=0.0001))])
,manifold,<abstractgrap...x7c4faa882350>
,n_selected_features,None
,"norm norm: {'l1', 'l2', 'max'}, default='l2'The norm to use to normalize each non zero sample. If norm='max'is used, values will be rescaled by the maximum of the absolutevalues.",'l2'
,"copy copy: bool, default=TrueSet to False to perform inplace row normalization and avoid acopy (if the input is already a numpy array or a scipy.sparseCSR matrix).",True
,"loss loss: {'hinge', 'log_loss', 'modified_huber', 'squared_hinge', 'perceptron', 'squared_error', 'huber', 'epsilon_insensitive', 'squared_epsilon_insensitive'}, default='hinge'The loss function to be used.- 'hinge' gives a linear SVM.- 'log_loss' gives logistic regression, a probabilistic classifier.- 'modified_huber' is another smooth loss that brings tolerance to outliers as well as probability estimates.- 'squared_hinge' is like hinge but is quadratically penalized.- 'perceptron' is the linear loss used by the perceptron algorithm.- The other losses, 'squared_error', 'huber', 'epsilon_insensitive' and 'squared_epsilon_insensitive' are designed for regression but can be useful in classification as well; see :class:`~sklearn.linear_model.SGDRegressor` for a description.More details about the losses formulas can be found in the :ref:`User Guide` and you can find a visualisation of the lossfunctions in:ref:`sphx_glr_auto_examples_linear_model_plot_sgd_loss_functions.py`.",'hinge'
,"penalty penalty: {'l2', 'l1', 'elasticnet', None}, default='l2'The penalty (aka regularization term) to be used. Defaults to 'l2'which is the standard regularizer for linear SVM models. 'l1' and'elasticnet' might bring sparsity to the model (feature selection)not achievable with 'l2'. No penalty is added when set to `None`.You can see a visualisation of the penalties in:ref:`sphx_glr_auto_examples_linear_model_plot_sgd_penalties.py`.",'l2'
,"alpha alpha: float, default=0.0001Constant that multiplies the regularization term. The higher thevalue, the stronger the regularization. Also used to compute thelearning rate when `learning_rate` is set to 'optimal'.Values must be in the range `[0.0, inf)`.",0.0001
,"l1_ratio l1_ratio: float, default=0.15The Elastic Net mixing parameter, with 0 <= l1_ratio <= 1.l1_ratio=0 corresponds to L2 penalty, l1_ratio=1 to L1.Only used if `penalty` is 'elasticnet'.Values must be in the range `[0.0, 1.0]` or can be `None` if`penalty` is not `elasticnet`... versionchanged:: 1.7 `l1_ratio` can be `None` when `penalty` is not ""elasticnet"".",0.15
,"fit_intercept fit_intercept: bool, default=TrueWhether the intercept should be estimated or not. If False, thedata is assumed to be already centered.",True


## 5. Fit

In [19]:
model.fit(X_train, y_train)
model

,transformer,"Pipeline(step..., nbits=14))])"
,estimator,Pipeline(step...tol=0.0001))])
,manifold,<abstractgrap...x7c4faa882350>
,n_selected_features,None
,"norm norm: {'l1', 'l2', 'max'}, default='l2'The norm to use to normalize each non zero sample. If norm='max'is used, values will be rescaled by the maximum of the absolutevalues.",'l2'
,"copy copy: bool, default=TrueSet to False to perform inplace row normalization and avoid acopy (if the input is already a numpy array or a scipy.sparseCSR matrix).",True
,"loss loss: {'hinge', 'log_loss', 'modified_huber', 'squared_hinge', 'perceptron', 'squared_error', 'huber', 'epsilon_insensitive', 'squared_epsilon_insensitive'}, default='hinge'The loss function to be used.- 'hinge' gives a linear SVM.- 'log_loss' gives logistic regression, a probabilistic classifier.- 'modified_huber' is another smooth loss that brings tolerance to outliers as well as probability estimates.- 'squared_hinge' is like hinge but is quadratically penalized.- 'perceptron' is the linear loss used by the perceptron algorithm.- The other losses, 'squared_error', 'huber', 'epsilon_insensitive' and 'squared_epsilon_insensitive' are designed for regression but can be useful in classification as well; see :class:`~sklearn.linear_model.SGDRegressor` for a description.More details about the losses formulas can be found in the :ref:`User Guide` and you can find a visualisation of the lossfunctions in:ref:`sphx_glr_auto_examples_linear_model_plot_sgd_loss_functions.py`.",'hinge'
,"penalty penalty: {'l2', 'l1', 'elasticnet', None}, default='l2'The penalty (aka regularization term) to be used. Defaults to 'l2'which is the standard regularizer for linear SVM models. 'l1' and'elasticnet' might bring sparsity to the model (feature selection)not achievable with 'l2'. No penalty is added when set to `None`.You can see a visualisation of the penalties in:ref:`sphx_glr_auto_examples_linear_model_plot_sgd_penalties.py`.",'l2'
,"alpha alpha: float, default=0.0001Constant that multiplies the regularization term. The higher thevalue, the stronger the regularization. Also used to compute thelearning rate when `learning_rate` is set to 'optimal'.Values must be in the range `[0.0, inf)`.",0.0001
,"l1_ratio l1_ratio: float, default=0.15The Elastic Net mixing parameter, with 0 <= l1_ratio <= 1.l1_ratio=0 corresponds to L2 penalty, l1_ratio=1 to L1.Only used if `penalty` is 'elasticnet'.Values must be in the range `[0.0, 1.0]` or can be `None` if`penalty` is not `elasticnet`... versionchanged:: 1.7 `l1_ratio` can be `None` when `penalty` is not ""elasticnet"".",0.15
,"fit_intercept fit_intercept: bool, default=TrueWhether the intercept should be estimated or not. If False, thedata is assumed to be already centered.",True


## 6. Test Metrics

In [ ]:
y_pred = model.predict(X_test)

metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
    "valid_held_out_evaluation": VALID_HELD_OUT_EVALUATION,
}
metrics_df = pd.Series(metrics, name="value").to_frame()
metrics_df

In [ ]:
report_labels = all_labels
report = classification_report(
    y_test,
    y_pred,
    labels=report_labels,
    output_dict=True,
    zero_division=0,
)
raw_report_df = pd.DataFrame(report).T
per_class_report_df = raw_report_df.loc[
    [label for label in report_labels if label in raw_report_df.index],
    ["precision", "recall", "f1-score", "support"],
].copy()

summary_rows = [
    {"metric": "accuracy", "value": float(report["accuracy"]), "support": len(y_test)},
    {"metric": "balanced_accuracy", "value": balanced_accuracy_score(y_test, y_pred), "support": len(y_test)},
]
for average_name, prefix in [("macro avg", "macro"), ("weighted avg", "weighted")]:
    average_report = report[average_name]
    summary_rows.extend(
        [
            {"metric": f"{prefix}_precision", "value": average_report["precision"], "support": average_report["support"]},
            {"metric": f"{prefix}_recall", "value": average_report["recall"], "support": average_report["support"]},
            {"metric": f"{prefix}_f1", "value": average_report["f1-score"], "support": average_report["support"]},
        ]
    )
summary_report_df = pd.DataFrame(summary_rows).set_index("metric")

report_df = per_class_report_df
display(per_class_report_df)
display(summary_report_df)

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=report_labels)
confusion_matrix_df = pd.DataFrame(
    cm,
    index=[f"true_{label}" for label in report_labels],
    columns=[f"pred_{label}" for label in report_labels],
)

cm_row_totals = cm.sum(axis=1, keepdims=True)
cm_fractions = cm / cm_row_totals.clip(min=1)
scale = 0.5
fig, ax = plt.subplots(figsize=(scale * len(report_labels) + 3, scale * len(report_labels) + 2), dpi=130)
im = ax.imshow(cm_fractions, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(report_labels)), labels=[str(label) for label in report_labels], rotation=45, ha="right")
ax.set_yticks(range(len(report_labels)), labels=[str(label) for label in report_labels])
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
ax.set_title("Confusion Matrix: count and true-class fraction")

for row_idx in range(cm.shape[0]):
    for col_idx in range(cm.shape[1]):
        fraction = cm_fractions[row_idx, col_idx]
        text_color = "white" if fraction >= 0.55 else "black"
        ax.text(
            col_idx,
            row_idx,
            f"{fraction:.1%}\n[#{cm[row_idx, col_idx]}]",
            ha="center",
            va="center",
            color=text_color,
            fontsize=9,
        )

fig.colorbar(im, ax=ax, label="fraction of true class")
fig.tight_layout()
plt.show()

confusion_matrix_df


## 7. Inspect Predictions

In [ ]:
predictions = metadata_for_split.iloc[test_idx][["sample_id", "pdb_id", "protein_name", "split_group", "first_level_ec_numbers", "graph_path"]].copy()
predictions["y_true"] = y_test
predictions["y_pred"] = y_pred
predictions["correct"] = predictions["y_true"] == predictions["y_pred"]
predictions.sort_values(["correct", "sample_id"]).head(30)

## 8. Persist Results

In [ ]:
persisted_results = persist_graph_estimator_results(
    results_root=RESULTS_ROOT,
    task=TASK,
    status_path=STATUS_PATH,
    predictions=predictions,
    metrics_df=metrics_df,
    report_df=report_df,
    summary_report_df=summary_report_df,
    confusion_matrix_df=confusion_matrix_df,
    graphs=graphs,
    graphs_for_split=graphs_for_split,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    split_strategy=split_strategy,
    split_group_column=SPLIT_GROUP_COLUMN,
    split_group_fallback_columns=SPLIT_GROUP_FALLBACK_COLUMNS,
    group_counts=group_counts,
    train_groups=train_groups,
    test_groups=test_groups,
    group_overlap=group_overlap,
    stratified=stratified,
    all_labels=all_labels,
    class_counts=class_counts,
    split_counts=split_counts,
    missing_from_train=missing_from_train,
    missing_from_test=missing_from_test,
    valid_held_out_evaluation=VALID_HELD_OUT_EVALUATION,
    min_samples_per_class_for_evaluation=MIN_SAMPLES_PER_CLASS_FOR_EVALUATION,
    filter_under_supported_classes=FILTER_UNDER_SUPPORTED_CLASSES,
    transformer_kind=TRANSFORMER_KIND,
    node_label_alphabet=NODE_LABEL_ALPHABET,
    transformer=transformer,
    base_estimator_kind=BASE_ESTIMATOR_KIND,
    model=model,
)
saved_files = persisted_results.to_frame()
saved_files
